# 04. Intensity-duration-frequency relationships

This notebook disaggregates the 24-h design depths of the observed baseline and the four future cases into shorter durations, converts them to intensities and fits a Sherman-type equation to each climate case. It produces the IDF curves and the table of Sherman parameters.


## Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
%matplotlib inline

ROOT = Path("..")
CASES = {   # climate case -> (label, design-rainfall table)
    "observed":         ("Observed",             ROOT / "results" / "02_frequency" / "design_rainfall_observed.csv"),
    "ssp245_2021_2050": ("SSP2-4.5 (2021-2050)", ROOT / "results" / "03_cmip6" / "design_rainfall_ssp245_2021_2050.csv"),
    "ssp245_2051_2080": ("SSP2-4.5 (2051-2080)", ROOT / "results" / "03_cmip6" / "design_rainfall_ssp245_2051_2080.csv"),
    "ssp585_2021_2050": ("SSP5-8.5 (2021-2050)", ROOT / "results" / "03_cmip6" / "design_rainfall_ssp585_2021_2050.csv"),
    "ssp585_2051_2080": ("SSP5-8.5 (2051-2080)", ROOT / "results" / "03_cmip6" / "design_rainfall_ssp585_2051_2080.csv"),
}
CHOSEN_DIST = "GEV"            # distribution selected in notebook 02
EXPONENT = 1.0 / 3.0           # exponent of the one-third rule
DURATIONS_HR = [1, 2, 3, 6, 12, 24]
OUTDIR = ROOT / "results" / "04_idf"
OUTDIR.mkdir(parents=True, exist_ok=True)

## 1. Method

The 24-h design depth $P_{24}$ (mm) is disaggregated to a duration of $t$ hours with the empirical reduction formula of the India Meteorological Department, the one-third rule (Andimuthu et al. 2019),

$$P_t = P_{24}\left(\frac{t}{24}\right)^{1/3}$$

and converted to the mean intensity $i = P_t/t$ (mm h$^{-1}$). A Sherman-type relationship (Koutsoyiannis et al. 1998) is then fitted to the intensities of all durations and return periods by non-linear least squares,

$$i = \frac{k\,T^{a}}{(d + \theta)^{m}}$$

where $T$ is the return period (years), $d$ the duration (h) and $k$, $a$, $\theta$ and $m$ the fitted coefficients. Because the one-third rule fixes the decline of intensity with duration, the fit returns $m = 2/3$ and $\theta = 0$; the equation summarises the curves in closed form and does not test them.

In [ ]:
def sherman(X, k, a, theta, m):
    T_, d_ = X
    return k * T_ ** a / (d_ + theta) ** m

## 2. IDF tables, curves and Sherman fit for each climate case

In [ ]:
col = f"{CHOSEN_DIST} (mm)"
sherman_rows = []

for case, (label, path) in CASES.items():
    design = pd.read_csv(path)
    if col not in design.columns:
        raise ValueError(f"Column '{col}' not found in {path}. Available: {list(design.columns)}")
    P24 = dict(zip(design["Return period (yr)"], design[col]))
    outdir = OUTDIR / case
    outdir.mkdir(parents=True, exist_ok=True)

    # depths and intensities by duration
    depth_rows, int_rows = [], []
    for T in sorted(P24):
        p24 = P24[T]
        drow = {"Return period (yr)": T}
        irow = {"Return period (yr)": T}
        for t in DURATIONS_HR:
            pt = p24 * (t / 24.0) ** EXPONENT
            drow[f"{t} h (mm)"] = round(pt, 2)
            irow[f"{t} h (mm/hr)"] = round(pt / t, 2)
        depth_rows.append(drow)
        int_rows.append(irow)
    pd.DataFrame(depth_rows).to_csv(outdir / "idf_depths.csv", index=False)
    pd.DataFrame(int_rows).to_csv(outdir / "idf_intensities.csv", index=False)

    # IDF curves
    fig, ax = plt.subplots(figsize=(8, 5.5))
    dfine = np.linspace(0.5, 24, 120)
    for T in sorted(P24):
        p24 = P24[T]
        ax.plot(dfine, p24 * (dfine / 24.0) ** EXPONENT / dfine, lw=1.8, label=f"{T}-year")
    for T in sorted(P24):
        p24 = P24[T]
        ax.scatter(DURATIONS_HR, [p24 * (t / 24.0) ** EXPONENT / t for t in DURATIONS_HR],
                   s=18, c="k", zorder=5)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Rainfall duration (hours)")
    ax.set_ylabel("Rainfall intensity (mm/hr)")
    ax.set_title(f"IDF curves: {label}")
    ax.legend(title="Return period", fontsize=8)
    ax.grid(alpha=0.35, which="both")
    fig.tight_layout()
    fig.savefig(outdir / "fig_idf_curves.png", dpi=300)
    plt.show()

    # Sherman equation
    Tg, Dg, Ig = [], [], []
    for T in sorted(P24):
        p24 = P24[T]
        for t in DURATIONS_HR:
            Tg.append(T); Dg.append(t); Ig.append(p24 * (t / 24.0) ** EXPONENT / t)
    Tg, Dg, Ig = np.array(Tg, float), np.array(Dg, float), np.array(Ig, float)

    popt, _ = curve_fit(sherman, (Tg, Dg), Ig, p0=[50, 0.2, 0.01, 0.7], maxfev=40000)
    k, a, theta, m = popt
    pred = sherman((Tg, Dg), *popt)
    r2 = 1 - np.sum((Ig - pred) ** 2) / np.sum((Ig - np.mean(Ig)) ** 2)
    sherman_rows.append({"Case": case, "Climate": label, "k": round(k, 4), "a": round(a, 4),
                         "theta": round(theta, 4) + 0.0, "m": round(m, 4), "R2": round(r2, 4)})

sherman_table = pd.DataFrame(sherman_rows)
sherman_table.to_csv(OUTDIR / "sherman_parameters.csv", index=False)
sherman_table